In [4]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [5]:
def fetch_data(year, month, day):
    url = f'https://sunspots.irsol.usi.ch/db/drawing.php?y={year}&m={month}&d={day}'
    response = requests.get(url)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        
        general_table = soup.find('div', class_='drawing-general-table')
        date_text = None
        time_text = None
        if general_table:
            date_text = general_table.find_all('div')[0].get_text(separator='|').strip().split('|')[0]
            time_text = general_table.find_all('div')[0].get_text(separator='|').strip().split('|')[1]
            #observer_text = general_table.find_all('div')[1].get_text(separator='|').strip().split('|')[0][11:]
        
        measures_table = soup.find('table', class_='measures-table')
        
        if measures_table:
            headers = [header.text for header in measures_table.find_all('th')]
            data = []
            rows = measures_table.find_all('tr')[1:] 
            
            for row in rows[:-2]:  # Исключаем последние две строки
                cols = row.find_all('td')
                data.append([date_text] + [time_text] + [col.text for col in cols])        
    
            return headers, data, None
        else:
            return None, None, f"Нет данных за {year}-{month}-{day}"

    
    else:
        return None, None, f"Ошибка при запросе данных за {year}-{month}-{day}: {response.status_code}"

In [6]:
# Выбор периода выгрузки данных
start_date = datetime(2010, 1, 1)
end_date = datetime(2020, 12, 31) 

all_data = []
headers = []
error_messages = []

current_date = start_date
while current_date <= end_date:
    year = current_date.year
    month = current_date.month
    day = current_date.day
    
    data_header, data, error_message = fetch_data(year, month, day)
    
    if error_message:  
        error_messages.append(error_message)
    
    if data:
        if not headers:  
            headers = ['Date'] + ['Time'] + data_header 
        all_data.extend(data)  

    current_date += timedelta(days=1)

df = pd.DataFrame(all_data, columns=headers)        
df.head()

,Date,Time,Group,Not Weighted,Weighted,Type,Latitude,Carrington Longit.
0,2010-01-01,10:00,27,6,9,D,-28.1,54.9
1,2010-01-02,08:30,27,10,15,D,-28.0,53.9
2,2010-01-04,09:00,27,2,7,G,-28.1,56.2
3,2010-01-09,10:00,1,4,3,B,28.8,238.2
4,2010-01-10,10:30,1,18,14,D,27.6,243.0


In [7]:
df.shape

(9975, 8)

In [109]:
df.to_csv('sunspot_drawing.csv', index=False)